# EXAFS Iterative Fitting Notebook

This notebook demonstrates how to use the `iterative_fit_all()` function from the XASmu2r class for EXAFS analysis. The workflow enables iteratively adding paths to your EXAFS model, evaluating their contribution, and improving your fit step by step.

**What this notebook does:**
- Sets up a modular workflow for iterative EXAFS fitting
- Integrates evaluation of candidate FEFF paths
- Provides visualizations at each step
- Implements a robust parameter tracking system across iterations

## Import Required Libraries

First, let's import all the necessary libraries for XAS analysis:

In [1]:
# Import core scientific libraries
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import copy
from scipy.signal import savgol_filter
import pandas as pd
from IPython.display import display, HTML

# Import Larch components for XAS analysis
from larch.io import read_athena
from larch.xafs import (pre_edge, autobk, xftf, ff2chi, feffpath, 
                       feffit_transform, feffit_dataset, feffit, 
                       feffit_report, TransformGroup, path2chi)
from larch.fitting import param, guess, param_group
from larch.utils import group2dict
from larch import Group

# Set up visualization preferences
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.style.use('ggplot')

# Configure Jupyter to show plots inline
%matplotlib inline

## Set Up XASmu2r Class and Data Structure

We'll create a streamlined version of the XASmu2r class with just the essential components needed for iterative fitting. This simplified version will make debugging and analysis easier in a notebook environment.

In [2]:
class XASmu2r_Lite:
    """
    A streamlined version of XASmu2r focused on iterative EXAFS fitting.
    """
    def __init__(self, prj_folder=None):
        """Initialize with a folder of Athena project files."""
        self.prj_folder = Path(prj_folder) if prj_folder else None
        self.projects = {}
        
    def load_projects(self, prj_folder=None):
        """Load Athena projects from a folder."""
        if prj_folder:
            self.prj_folder = Path(prj_folder)
        
        if not self.prj_folder or not self.prj_folder.exists():
            print(f"Project folder not found: {self.prj_folder}")
            return
            
        projects = {}
        # Load projects from all .prj files found in the folder
        for prj_file in self.prj_folder.glob("*.prj"):
            print(f"Loading project file: {prj_file.name}")
            project = read_athena(prj_file)
            projects[prj_file.stem] = project

        # Print summary of loaded groups
        for proj_name, project in projects.items():
            print(f"\nAvailable Groups in Project '{proj_name}':")
            for group_name in project.groups:
                print(f"  {group_name}")
                
        self.projects = projects
        return projects
    
    def get_transform_for_group(self, data_group):
        """
        Create and return a TransformGroup for a data group.
        """
        print(f"Creating Fourier Transform parameters for {getattr(data_group, 'group_name', 'unknown')}...")
        kw = 3
        kmin = 3.0
        kmax = 12.0
        
        # Use optimized parameters if available
        if hasattr(data_group, 'best_kmin_kw3'):
            kmin = data_group.best_kmin_kw3
            print(f"  Found optimized kmin = {kmin:.2f} for kw=3")
        else:
            print(f"  Using default kmin = {kmin:.2f}")
            
        if hasattr(data_group, 'best_kmax_kw3'):
            kmax = data_group.best_kmax_kw3
            print(f"  Found optimized kmax = {kmax:.2f} for kw=3")
        else:
            print(f"  Using default kmax = {kmax:.2f}")
            
        # Get rbkg value if available
        rbkg = getattr(data_group, 'rbkg', 0.9)
        if rbkg is None:
            print("  Warning: No rbkg value found, using default 0.9")
            rbkg = 0.9
            
        # Create transform group
        trans = TransformGroup(kmin=kmin, kmax=kmax, kweight=kw, 
                              dk=3, window='hanning', 
                              rmin=rbkg, rmax=4.0)
        
        print(f"  Created transform with k-range: {kmin:.2f} to {kmax:.2f} Å⁻¹, kw={kw}")
        return trans

## Prepare XAS Data and FEFF Paths

Now we'll define functions to prepare our XAS data and organize FEFF calculation paths. This includes functions to analyze and categorize different types of paths (Pb-I, Pb-O, etc.) and extract relevant information.

In [3]:
def analyze_feff_path(file_path, category):
    """
    Analyze a FEFF path file for a specific category (e.g., Pb-O).
    
    Parameters:
        file_path (str): Path to the FEFF .dat file
        category (str): Category to analyze for (e.g., "Pb-O")
        
    Returns:
        dict: Path information including whether it matches the category
    """
    path_info = {f'is_{category.lower()}': False, 
                'path': file_path,
                'filename': os.path.basename(file_path)}
    
    try:
        # Read the FEFF path
        fp_obj = feffpath(filename=file_path)
        
        # Check absorber is Pb
        if hasattr(fp_obj._feffdat, 'absorber') and fp_obj._feffdat.absorber != 'Pb':
            return path_info
            
        # Extract target element from category (e.g., "O" from "Pb-O")
        target_element = None
        if '-' in category:
            parts = category.split('-')
            if len(parts) >= 2:
                target_element = parts[1]
        else:
            target_element = category
            
        if not target_element:
            return path_info
            
        # Analyze scatterers
        scatterers = []
        target_atoms = []
        for atom in fp_obj._feffdat.geom:
            atom_symbol = atom[0]
            if atom[2] != 0:  # not the absorber
                scatterers.append(atom_symbol)
                if atom_symbol.lower() == target_element.lower():
                    target_atoms.append(atom)
                    
        if target_atoms:
            path_info[f'is_{category.lower()}'] = True
            path_info['reff'] = fp_obj._feffdat.reff
            path_info['nleg'] = fp_obj._feffdat.nleg
            path_info['degen'] = fp_obj._feffdat.degen
            path_info['geometry'] = fp_obj._feffdat.geom
            path_info[f'n_{category.lower()}'] = scatterers.count(target_element)
            path_info['feffpath'] = fp_obj
            
            # Determine path type
            if fp_obj._feffdat.nleg == 2 and len(target_atoms) == 1:
                path_info['path_type'] = 'ss'  # single scattering
            elif fp_obj._feffdat.nleg > 2:
                path_info['path_type'] = 'ms'  # multiple scattering
                
            # Create a descriptive label
            if path_info.get('path_type') == 'ss':
                path_info['label'] = f"Pb-{target_element} SS: {path_info['reff']:.3f}Å"
            else:
                # For multiple scattering, count atoms of each type
                atoms_count = {}
                for a in scatterers:
                    atoms_count[a] = atoms_count.get(a, 0) + 1
                atom_string = '-'.join([f"{cnt}{atom}" for atom, cnt in atoms_count.items()])
                path_info['label'] = f"Pb-{atom_string} MS: {path_info['reff']:.3f}Å"
                
        return path_info
    
    except Exception as e:
        print(f"Error analyzing {file_path}: {e}")
        return path_info

def get_candidate_paths(feff_base_dir, category, max_reff=4.0):
    """
    Find and filter FEFF paths for a given category.
    
    Parameters:
        feff_base_dir (str or Path): Directory with FEFF calculations
        category (str): Category to search for (e.g., "Pb-O")
        max_reff (float): Maximum effective radius to consider
        
    Returns:
        list: Filtered paths matching criteria
    """
    feff_base_dir = Path(feff_base_dir)
    print(f"Searching for candidate '{category}' paths in {feff_base_dir}...")
    
    candidate_infos = []
    for root, dirs, files in os.walk(feff_base_dir):
        for file in files:
            if file.lower().endswith('.dat') and 'feff' in file.lower():
                full_path = os.path.join(root, file)
                info = analyze_feff_path(full_path, category)
                if info.get(f'is_{category.lower()}'):
                    candidate_infos.append(info)
    
    # Sort by effective distance
    candidate_infos.sort(key=lambda x: x.get('reff', float('inf')))
    print(f"Found {len(candidate_infos)} candidate paths for category '{category}'.")
    
    # Filter to relevant single-scattering paths with reff < max_reff
    filtered = [p for p in candidate_infos if 
               p.get('path_type')=='ss' and 
               p.get('reff', 0) < max_reff]
    
    # If no SS paths, use first MS path as fallback
    if len(filtered) == 0 and candidate_infos:
        ms_paths = [p for p in candidate_infos if 
                   p.get('path_type')=='ms' and 
                   p.get('reff',0) < max_reff]
        if ms_paths:
            filtered.append(ms_paths[0])
            
    print(f"Filtered to {len(filtered)} most relevant {category} paths.")
    
    return filtered

## Define Helper Functions for Path Analysis

Let's create helper functions to categorize paths (single scattering vs. multiple scattering) and set up appropriate parameter relationships for our EXAFS model.

In [4]:
def categorize_ss_ms_paths(paths):
    """
    Categorize paths into single-scattering (SS) and multiple-scattering (MS) paths.
    
    Parameters:
        paths (list): List of FEFF path objects
            
    Returns:
        tuple: (ss_paths, ms_paths) - dictionaries of paths by category
    """
    ss_paths = {}  # Category -> list of SS paths
    ms_paths = {}  # Category -> list of MS paths
    
    for path in paths:
        # Determine category from path label
        category = 'unknown'
        if 'Pb-I' in path.label:
            category = 'pbi'
        elif 'Pb-O' in path.label:
            category = 'pbo'
        elif 'Pb-S' in path.label:
            category = 'pbs'
        elif 'Pb-Cs' in path.label:
            category = 'pbcs'
        
        # Determine if SS or MS path
        is_ms = False
        if hasattr(path, 'nleg'):
            is_ms = path.nleg > 2
        elif 'MS' in path.label:
            is_ms = True
            
        # Add to appropriate dictionary
        if is_ms:
            if category not in ms_paths:
                ms_paths[category] = []
            ms_paths[category].append(path)
        else:
            if category not in ss_paths:
                ss_paths[category] = []
            ss_paths[category].append(path)
    
    return ss_paths, ms_paths

def apply_parameters_to_paths(all_paths, ss_paths, ms_paths):
    """
    Apply parameter names to each path based on its category and type.
    
    Parameters:
        all_paths (list): All paths to be used
        ss_paths (dict): Dictionary of SS paths by category
        ms_paths (dict): Dictionary of MS paths by category
    """
    # Apply standard parameters to SS paths
    for category, paths in ss_paths.items():
        for path in paths:
            path.s02 = f"amp * n_{category}"
            path.e0 = "del_e0"
            path.sigma2 = f"{category}_sig2"
            path.deltar = f"{category}_delr"
            
    # Apply parameters to MS paths
    for category, paths in ms_paths.items():
        for i, path in enumerate(paths):
            # Set standard parameters
            path.s02 = f"amp * n_{category}"  # Using same amplitude as SS
            path.e0 = "del_e0"
            
            # Set sigma2 for MS paths - typically related to the SS path sigma2
            path.sigma2 = f"{category}_sig2_ms{i+1}"
            
            # For deltar, use separate parameters for MS paths
            path.deltar = f"{category}_ms{i+1}_delr"

def create_fit_parameters(iteration, category, prev_fit=None, fixed_params=False):
    """
    Create parameter group for fitting at a given iteration.
    
    Parameters:
        iteration (int): Current iteration number
        category (str): Path category being evaluated (e.g., 'Pb-O')
        prev_fit (object): Previous fit result to extract parameters from
        fixed_params (bool): Whether to fix parameters from previous fit
        
    Returns:
        param_group: Parameters configured for fitting
    """
    if prev_fit is not None and hasattr(prev_fit, 'params'):
        # Extract values from previous fit
        prev_params = prev_fit.params
        amp_value = prev_params['amp'].value if 'amp' in prev_params else 0.94
        del_e0_value = prev_params['del_e0'].value if 'del_e0' in prev_params else 0.0
        n_PbI_value = prev_params['n_PbI'].value if 'n_PbI' in prev_params else 3.0
        pbi_sig2_value = prev_params['pbi_sig2'].value if 'pbi_sig2' in prev_params else 0.01
        pbi_delr_value = prev_params['pbi_delr'].value if 'pbi_delr' in prev_params else 0.0
    else:
        # Default values if no previous fit
        amp_value = 0.94
        del_e0_value = 0.0
        n_PbI_value = 3.0
        pbi_sig2_value = 0.01
        pbi_delr_value = 0.0
    
    # Create parameter group
    fit_params = param_group(
        amp = param(amp_value, vary=False, min=0.5, max=1.2),
        del_e0 = param(del_e0_value, vary=not fixed_params, min=-15, max=15),
        n_PbI = param(n_PbI_value, vary=not fixed_params, min=0.0, max=8.0),
        pbi_sig2 = param(pbi_sig2_value, vary=not fixed_params, min=0.001, max=0.03),
        pbi_delr = param(pbi_delr_value, vary=not fixed_params, min=-0.2, max=0.2)
    )
    
    # Add category-specific parameters for new path
    if category.lower() == 'pb-o':
        cat_prefix = 'pbo'
    elif category.lower() == 'pb-s':
        cat_prefix = 'pbs'  
    else:
        cat_prefix = category.lower().replace('-', '')
        
    # For test fits with fixed previous parameters, we want the new path params to vary
    fit_params.__setattr__(f"n_{cat_prefix}", param(1.0, vary=True, min=0.0, max=8.0))
    fit_params.__setattr__(f"{cat_prefix}_sig2", param(0.01, vary=True, min=0.001, max=0.03))
    fit_params.__setattr__(f"{cat_prefix}_delr", param(0.0, vary=True, min=-0.2, max=0.2))
    
    return fit_params

## Implement the Iterative Fitting Workflow

Now let's implement our modular version of the `iterative_fit_all()` function. We'll break it down into smaller components that can be executed and monitored cell by cell:

1. First, we'll define the main function that orchestrates the iterative fitting process
2. Then, we'll implement the process for evaluating and selecting paths
3. Finally, we'll create a step-by-step workflow that's easy to follow

In [12]:
def enhanced_iteration1_fit(projects, feff_base_dir=None):
    """
    Enhanced Iteration 1 – Evaluate and select optimal Pb-O path for all groups with residual analysis.
    
    Parameters:
    -----------
    projects : dict
        Dictionary of projects loaded and processed from Athena
    feff_base_dir : str or Path, optional
        Directory containing FEFF calculation files 
        (default: 'C:/Users/<user>/.larch/feff/relaxed')
    
    This function:
    - Walks the FEFF base directory to collect and analyze FEFF .dat files
    - Filters for relevant Pb-O paths
    - For each group with required data (k and chi_tapered):
        - Creates a transform
        - Evaluates each filtered FEFF path by performing test fits
        - Performs residual analysis in k-space within the FT window
        - Selects the best path using comprehensive quality metrics
        - Performs a final fit with detailed residual analysis
        - Stores results on the group and displays diagnostics
    
    Returns:
    --------
    dict
        The modified projects dictionary
    """
    import os
    from pathlib import Path
    import copy
    import numpy as np
    import matplotlib.pyplot as plt
    from larch.xafs import feffpath, feffit_dataset, feffit, TransformGroup
    from larch.fitting import param, param_group
    from larch import Group
    
    # Check if projects is valid
    if not projects:
        print("ERROR: No projects provided or projects is None/empty")
        return projects  # Return the original projects (even if None) instead of None
    
    print(f"Starting enhanced_iteration1_fit with {len(projects)} projects")
    
    # Use default FEFF directory if not provided
    if feff_base_dir is None:
        import os
        username = os.environ.get('USERNAME') or os.environ.get('USER')
        feff_base_dir = f"C:/Users/{username}/.larch/feff/relaxed"
    
    print(f"Searching for and analyzing Pb-O FEFF paths in {feff_base_dir}")
    
    # --- Inner helper: analyze a FEFF path file ---
    def analyze_feff_path(file_path):
        path_info = {'is_pbo': False, 'path': file_path, 'filename': os.path.basename(file_path)}
        try:
            fp_obj = feffpath(filename=file_path)
            if hasattr(fp_obj._feffdat, 'absorber') and fp_obj._feffdat.absorber != 'Pb':
                return path_info
            scatterers = []
            oxygen_atoms = []
            for atom in fp_obj._feffdat.geom:
                atom_symbol = atom[0]
                if atom[2] != 0:  # not the absorber
                    scatterers.append(atom_symbol)
                    if atom_symbol == 'O':
                        oxygen_atoms.append(atom)
            if 'O' in scatterers:
                path_info['is_pbo'] = True
                path_info['reff'] = fp_obj._feffdat.reff
                path_info['nleg'] = fp_obj._feffdat.nleg
                path_info['degen'] = fp_obj._feffdat.degen
                path_info['geometry'] = fp_obj._feffdat.geom
                path_info['n_oxygen'] = scatterers.count('O')
                path_info['feffpath'] = fp_obj
                if fp_obj._feffdat.nleg == 2 and len(oxygen_atoms) == 1:
                    path_info['path_type'] = 'ss'
                elif fp_obj._feffdat.nleg > 2:
                    path_info['path_type'] = 'ms'
                if path_info.get('path_type') == 'ss':
                    path_info['label'] = f"Pb-O SS: {path_info['reff']:.3f}Å"
                else:
                    atoms_count = {}
                    for atom in scatterers:
                        atoms_count[atom] = atoms_count.get(atom, 0) + 1
                    atom_string = '-'.join([f"{cnt}{atom}" for atom, cnt in atoms_count.items()])
                    path_info['label'] = f"Pb-{atom_string} MS: {path_info['reff']:.3f}Å"
            return path_info
        except Exception as e:
            print(f"Error analyzing {file_path}: {e}")
            return path_info
    
    # --- New helper: calculate residual metrics ---
    def calculate_residuals(dset, kweight=2):
        """Calculate residuals and metrics between data and model in k-space."""
        try:
            # Extract k, data chi and model chi
            k = dset.data.k
            chi_data = dset.data.chi
            chi_model = dset.model.chi
            
            # Get mask for points within the FT window
            ft_mask = (k >= dset.transform.kmin) & (k <= dset.transform.kmax)
            k_in_range = k[ft_mask]
            
            # Apply k-weighting to make residuals match visual assessment
            kw_factor = k_in_range**kweight
            data_weighted = chi_data[ft_mask] * kw_factor
            model_weighted = chi_model[ft_mask] * kw_factor
            
            # Calculate residuals and metrics
            residuals = data_weighted - model_weighted
            residuals_raw = chi_data[ft_mask] - chi_model[ft_mask]
            
            rmse = np.sqrt(np.mean(residuals**2))
            mean_abs_error = np.mean(np.abs(residuals))
            max_error = np.max(np.abs(residuals))
            
            return {
                'k': k_in_range,
                'residuals': residuals,
                'residuals_raw': residuals_raw,
                'rmse': rmse,
                'mean_abs_error': mean_abs_error,
                'max_error': max_error,
                'data_weighted': data_weighted,
                'model_weighted': model_weighted
            }
        except Exception as e:
            print(f"Error calculating residuals: {e}")
            return {
                'k': np.array([]),
                'residuals': np.array([]),
                'residuals_raw': np.array([]),
                'rmse': 999.0,
                'mean_abs_error': 999.0,
                'max_error': 999.0,
                'data_weighted': np.array([]),
                'model_weighted': np.array([])
            }
    
    # --- Collect all available Pb-O paths ---
    try:
        feff_base_dir = Path(feff_base_dir)
        if not feff_base_dir.exists():
            print(f"ERROR: FEFF directory {feff_base_dir} does not exist")
            return projects
            
        pbo_path_infos = []
        for root, dirs, files in os.walk(feff_base_dir):
            for file in files:
                if file.lower().endswith('.dat') and 'feff' in file.lower():
                    full_path = os.path.join(root, file)
                    info = analyze_feff_path(full_path)
                    if info.get('is_pbo'):
                        pbo_path_infos.append(info)
        pbo_path_infos.sort(key=lambda x: x.get('reff', 0))
        print(f"Found {len(pbo_path_infos)} Pb-O FEFF paths.")
        
        # Display basic summary of paths
        if pbo_path_infos:
            print("\nDetailed information about Pb-O paths:")
            print(f"{'Index':6} {'Type':6} {'Reff':8} {'Deg':6} {'n-leg':6} {'Label'}")
            print("-"*80)
            for i, info in enumerate(pbo_path_infos):
                ptype = info.get('path_type', 'unk')
                reff = info.get('reff', 0)
                deg = info.get('degen', 0)
                nleg = info.get('nleg', 0)
                label = info.get('label', info.get('filename', 'unknown'))
                print(f"{i+1:6d} {ptype:6s} {reff:.3f} Å {deg:6g} {nleg:6d}  {label}")
        else:
            print("WARNING: No Pb-O paths found in FEFF directory")
            return projects  # Return original projects if no paths found
    except Exception as e:
        print(f"ERROR collecting FEFF paths: {e}")
        return projects
    
    # Filter to relevant single-scattering paths with reff < 4.0 Å
    filtered_paths = [p for p in pbo_path_infos if p.get('path_type')=='ss' and p.get('reff',0) < 3.5]
    if len(filtered_paths) == 0:
        ms_paths = [p for p in pbo_path_infos if p.get('path_type')=='ms' and p.get('reff',0) < 4.0]
        if ms_paths:
            filtered_paths.append(ms_paths[0])
    print(f"\nFiltered to {len(filtered_paths)} most relevant Pb-O paths for testing with each group.")
    
    if not filtered_paths:
        print("ERROR: No suitable Pb-O paths found after filtering")
        return projects
    
    # Helper function to get transform for a group
    def get_transform_for_group(data_group):
        """Create and return a TransformGroup for the given data_group."""
        kw = 3
        kmin = 3.0
        kmax = 12.0
        if hasattr(data_group, 'best_kmin_kw3'):
            kmin = data_group.best_kmin_kw3
            print(f"  Found optimized kmin = {kmin:.2f} for kw=3")
        else:
            print(f"  Using default kmin = {kmin:.2f}")
        if hasattr(data_group, 'best_kmax_kw3'):
            kmax = data_group.best_kmax_kw3
            print(f"  Found optimized kmax = {kmax:.2f} for kw=3")
        else:
            print(f"  Using default kmax = {kmax:.2f}")
            
        rbkg = getattr(data_group, 'optimal_rbkg', getattr(data_group, 'rbkg', 0.9))
        if rbkg is None:
            print("  Warning: No rbkg value found, using default 0.9")
            rbkg = 0.9
        trans = TransformGroup(kmin=kmin, kmax=kmax, kweight=kw, dk=3, window='hanning', rmin=rbkg, rmax=4.0)
        print(f"  Created transform with k-range: {kmin:.2f} to {kmax:.2f} Å⁻¹, kw={kw}")
        return trans
    
    # --- Inner helper: process a single group ---
    def process_group(group, group_name):
        """Process a single group to find and fit the best Pb-O path."""
        print("\n" + "="*50)
        print(f"Processing group: {group_name}")
        print("="*50)
        if not (hasattr(group, 'k') and hasattr(group, 'chi_tapered')):
            print(f"Skipping {group_name} - missing required k and chi_tapered data")
            return False
        try:
            trans = get_transform_for_group(group)
            temp_data = Group(k=group.k, chi=group.chi_tapered)
        except Exception as e:
            print(f"Error creating transform for {group_name}: {e}")
            return False
            
        print(f"Evaluating {len(filtered_paths)} Pb-O paths for {group_name}...")
        single_path_results = []
        for i, path_info in enumerate(filtered_paths):
            try:
                fp_obj = path_info['feffpath']
                reff = path_info.get('reff', 0)
                path_label = path_info.get('label', 'unknown')
                
                # Create test path
                test_path = feffpath(fp_obj.filename)
                test_path.degen = 1.0
                test_path.s02 = 'amp * n_PbO'
                test_path.e0 = 'del_e0'
                test_path.sigma2 = 'pbo_sig2'
                test_path.deltar = 'pbo_delr'
                test_path.label = path_label
                
                # Create parameters
                test_params = build_params_for_iteration(1)
                
                # Create dataset and perform fit
                dset = feffit_dataset(data=temp_data, pathlist=[test_path], transform=trans)
                fit_result = feffit(test_params, [dset])
                
                # Calculate residuals and metrics
                residual_analysis = calculate_residuals(dset, kweight=trans.kweight)
                
                # Calculate quality metrics (including residual information)
                delr_abs = abs(fit_result.params['pbo_delr'].value)
                rmse_factor = residual_analysis['rmse'] / 50  # Scale RMSE to be comparable with other factors
                quality_metric = (delr_abs + fit_result.rfactor + rmse_factor)**2
                
                # Store result
                single_path_results.append({
                    'path_info': path_info,
                    'path': test_path,
                    'reff': reff,
                    'rfactor': fit_result.rfactor,
                    'n_PbO': fit_result.params['n_PbO'].value,
                    'pbo_sig2': fit_result.params['pbo_sig2'].value,
                    'pbo_delr': fit_result.params['pbo_delr'].value,
                    'chi_square': fit_result.chi_square,
                    'fit_result': fit_result,
                    'label': path_label,
                    'path_type': path_info.get('path_type', 'unk'),
                    'delr_abs': delr_abs,
                    'residual_analysis': residual_analysis,
                    'rmse': residual_analysis['rmse'],
                    'quality_metric': quality_metric,
                    'dataset': dset
                })
                print(f"  Path {i+1}: {path_label} - |ΔR|: {delr_abs:.3f}, R-factor: {fit_result.rfactor:.6f}, RMSE: {residual_analysis['rmse']:.3e}, Quality: {quality_metric:.6f}")
            except Exception as e:
                print(f"  Error testing path {path_info.get('label', 'unknown')} with {group_name}: {e}")
        
        if not single_path_results:
            print(f"No successful path fits for {group_name}")
            return False
            
        # Sort by quality metric and get best result
        single_path_results.sort(key=lambda x: x['quality_metric'])
        best_path_result = single_path_results[0]
        print("\nBest path results for", group_name)
        print(f"Path: {best_path_result['label']}")
        print(f"Quality metric: {best_path_result['quality_metric']:.6f}")
        print(f"|ΔR|: {best_path_result['delr_abs']:.6f}")
        print(f"R-factor: {best_path_result['rfactor']:.8f}")
        print(f"RMSE in k-space: {best_path_result['rmse']:.3e}")
        print(f"N(Pb-O): {best_path_result['n_PbO']:.2f}")
        print(f"Pb-O sigma2: {best_path_result['pbo_sig2']:.6f}")
        
        # Perform final fit with the best path
        path_info = best_path_result['path_info']
        fp_obj = path_info['feffpath']
        pbo_path = feffpath(fp_obj.filename)
        pbo_path.degen = 1.0
        pbo_path.s02 = 'amp * n_PbO'
        pbo_path.e0 = 'del_e0'
        pbo_path.sigma2 = 'pbo_sig2'
        pbo_path.deltar = 'pbo_delr'
        pbo_path.label = best_path_result['label']
        
        final_params = param_group(
            amp = param(0.78, vary=False, min=0.5, max=1.2),
            del_e0 = param(best_path_result['fit_result'].params['del_e0'].value, vary=True, min=-15, max=15),
            n_PbO = param(best_path_result['n_PbO'], vary=True, min=0.0, max=8.0),
            pbo_sig2 = param(best_path_result['pbo_sig2'], vary=True, min=0.001, max=0.03),
            pbo_delr = param(best_path_result['pbo_delr'], vary=True, min=-0.2, max=0.2)
        )
        
        final_dset = feffit_dataset(data=temp_data, pathlist=[pbo_path], transform=trans)
        fit_result_iter1 = feffit(final_params, final_dset)
        pbo_r_eff = pbo_path.reff + fit_result_iter1.params['pbo_delr'].value
        
        # Calculate final residuals
        final_residuals = calculate_residuals(final_dset, kweight=trans.kweight)
        
        print(f"Final fit results:")
        print(f"  R-factor = {fit_result_iter1.rfactor:.6f}")
        print(f"  N(Pb-O) = {fit_result_iter1.params['n_PbO'].value:.2f}")
        print(f"  |ΔR| = {abs(fit_result_iter1.params['pbo_delr'].value):.6f}")
        print(f"  Effective R = {pbo_r_eff:.3f} Å")
        print(f"  RMSE in k-space = {final_residuals['rmse']:.3e}")
        print(f"  Mean absolute error = {final_residuals['mean_abs_error']:.3e}")
        print(f"  Maximum error = {final_residuals['max_error']:.3e}")
        
        # Store results in the group
        group.fit_result_iter1 = fit_result_iter1
        group.paths_iter1 = [pbo_path]
        group.residual_analysis_iter1 = final_residuals
        
        # Extract and store k-space and r-space data for visualization
        dset0 = fit_result_iter1.datasets[0]
        k = dset0.data.k
        chi_data = dset0.data.chi * k**3
        chi_fit = dset0.model.chi * k**3
        r = dset0.data.r
        chir_data_mag = np.sqrt(dset0.data.chir_re**2 + dset0.data.chir_im**2)
        chir_fit_mag = np.sqrt(dset0.model.chir_re**2 + dset0.model.chir_im**2)
        
        group.k_space_iter1 = {'k': k, 'data': chi_data, 'fit': chi_fit}
        group.r_space_iter1 = {'r': r, 'data': chir_data_mag, 'fit': chir_fit_mag}
        
        # Create visualization with residuals
        create_visualizations(
            group, group_name, pbo_path, fit_result_iter1, dset0, 
            k, r, chi_data, chi_fit, chir_data_mag, chir_fit_mag,
            final_residuals
        )
        return True
def iterative_fit_all(xasmu2r, feff_category, iteration, feff_base_dir, previous_params=None):
    """
    Perform iterative EXAFS fitting for all groups by evaluating candidate paths.
    
    Parameters:
        xasmu2r: XASmu2r or XASmu2r_Lite instance with loaded projects
        feff_category (str): The path category to evaluate (e.g., "Pb-O")
        iteration (int): Current iteration number (e.g., 2)
        feff_base_dir (str or Path): Directory containing FEFF calculations
        previous_params (dict, optional): Dict of parameters from previous iterations
        
    Returns:
        dict: Updated parameters for each group with fit results
    """
    # Initialize return dictionary to store parameters for next iteration
    return_params = {}
    
    # Summary tracking
    total_processed = 0
    summary = []
    
    # Loop through all projects and groups
    for proj_name, project in xasmu2r.projects.items():
        print(f"\nProcessing project: {proj_name}")
        
        for gname, group in project.groups.items():
            group_id = f"{proj_name}.{gname}"
            print(f"\nProcessing group: {group_id}")
            
            # Check if group has required data
            if not (hasattr(group, 'k') and hasattr(group, 'chi_tapered')):
                print(f"  Skipping {group_id} – missing k or chi_tapered data.")
                continue

            # Get previous iteration result
            prev_fit_attr = f"fit_result_iter{iteration-1}"
            prev_fit = getattr(group, prev_fit_attr, None)
            if prev_fit is None:
                print(f"  Skipping {group_id} – no previous iteration result ({prev_fit_attr}).")
                continue
                
            print(f"  Found previous iteration result for {group_id}.")
            
            # Store the previous R-factor for comparison
            prev_rfactor = prev_fit.rfactor
            print(f"  Previous iteration R-factor: {prev_rfactor:.6f}")

            # Extract parameters from previous fit
            try:
                # Always use the original amp value from iteration 1 if available
                fit_result_iter1 = getattr(group, "fit_result_iter1", None)
                if fit_result_iter1 is not None and hasattr(fit_result_iter1, 'params'):
                    amp_value = fit_result_iter1.params['amp'].value
                    print(f"  Using amplitude value {amp_value:.4f} from iteration 1")
                else:
                    amp_value = prev_fit.params['amp'].value
                    print(f"  Using amplitude value {amp_value:.4f} from previous iteration")
                
                # Get other parameters from previous iteration or provided parameters
                if previous_params and group_id in previous_params:
                    group_params = previous_params[group_id]
                    print(f"  Using provided parameters for {group_id}")
                    del_e0_value = group_params.get('del_e0', prev_fit.params['del_e0'].value)
                    n_PbI_value = group_params.get('n_PbI', prev_fit.params['n_PbI'].value)
                    pbi_sig2_value = group_params.get('pbi_sig2', prev_fit.params['pbi_sig2'].value)
                    pbi_delr_value = group_params.get('pbi_delr', prev_fit.params['pbi_delr'].value)
                else:
                    del_e0_value = prev_fit.params['del_e0'].value
                    n_PbI_value = prev_fit.params['n_PbI'].value
                    pbi_sig2_value = prev_fit.params['pbi_sig2'].value
                    pbi_delr_value = prev_fit.params['pbi_delr'].value
                
                # Initialize entry in return_params with parameters
                return_params[group_id] = {
                    'amp': amp_value,
                    'del_e0': del_e0_value,
                    'n_PbI': n_PbI_value,
                    'pbi_sig2': pbi_sig2_value,
                    'pbi_delr': pbi_delr_value,
                    'rfactor': prev_rfactor,
                    'paths': [],
                    'iteration': iteration-1
                }
                
                print(f"  Parameters: amp={amp_value:.4f}, del_e0={del_e0_value:.4f}, n_PbI={n_PbI_value:.4f}")
            except Exception as e:
                print(f"  Error extracting parameters: {e}")
                continue
                
            # Process this individual group
            success, group_result = process_group_iteration(
                group, group_id, 
                xasmu2r, 
                feff_category, 
                iteration, 
                feff_base_dir,
                prev_fit
            )
            
            if success:
                # Update results
                total_processed += 1
                
                # Update return_params with new fit parameters if R-factor improved
                if group_result['improved']:
                    final_fit = group_result['fit_result']
                    return_params[group_id].update({
                        'amp': final_fit.params['amp'].value,
                        'del_e0': final_fit.params['del_e0'].value,
                        'n_PbI': final_fit.params['n_PbI'].value,
                        'pbi_sig2': final_fit.params['pbi_sig2'].value,
                        'pbi_delr': final_fit.params['pbi_delr'].value,
                        'rfactor': final_fit.rfactor,
                        'iteration': iteration
                    })
                    
                    # Add new path category params
                    cat_prefix = feff_category.lower().replace('-', '')
                    if f"n_{cat_prefix}" in final_fit.params:
                        return_params[group_id][f"n_{cat_prefix}"] = final_fit.params[f"n_{cat_prefix}"].value
                    if f"{cat_prefix}_sig2" in final_fit.params:
                        return_params[group_id][f"{cat_prefix}_sig2"] = final_fit.params[f"{cat_prefix}_sig2"].value
                    if f"{cat_prefix}_delr" in final_fit.params:
                        return_params[group_id][f"{cat_prefix}_delr"] = final_fit.params[f"{cat_prefix}_delr"].value
                    
                    # Add new path info
                    new_path = group_result.get('new_path')
                    if new_path:
                        return_params[group_id]['paths'].append({
                            'label': new_path.label,
                            'reff': new_path.reff,
                            'category': feff_category
                        })
                
                # Add to summary
                summary.append(group_result['summary'])
    
    # Print summary
    print("\n" + "="*80)
    print(f"ITERATION {iteration} SUMMARY")
    print("="*80)
    print(f"Processed {total_processed} groups")
    
    if summary:
        # Create a DataFrame for better display
        df = pd.DataFrame(summary)
        display(HTML(df.to_html(index=False)))
    
    return return_params

## Process and Evaluate Candidate Paths

Now let's implement the function to process an individual group during the iterative fitting. This function evaluates candidate paths and selects the best one to add to the current model.

In [6]:
def process_group_iteration(group, group_id, xasmu2r, feff_category, iteration, feff_base_dir, prev_fit):
    """
    Process a single group for one iteration of EXAFS fitting.
    
    Parameters:
        group: The data group to process
        group_id: String identifier for the group
        xasmu2r: The XASmu2r instance
        feff_category: Category of paths to evaluate
        iteration: Current iteration number
        feff_base_dir: Directory with FEFF calculations
        prev_fit: Previous iteration fit result
        
    Returns:
        tuple: (success, result_dict) with detailed fit information
    """
    # Create transform and data group
    trans = xasmu2r.get_transform_for_group(group)
    # For iteration > 1, extend R-range slightly
    if iteration > 1:
        trans.rmax = trans.rmax + 0.2 * (iteration-1)
        print(f"  Extended R-range to {trans.rmax:.2f} Å for iteration {iteration}")
        
    temp_data = Group(k=group.k, chi=group.chi_tapered)
    
    # Get previous paths
    prev_paths_attr = f"paths_iter{iteration-1}"
    prev_paths = getattr(group, prev_paths_attr, [])
    if not prev_paths:
        print(f"  Error: No paths from previous iteration for {group_id}, skipping.")
        return False, {}
    
    print(f"  Using {len(prev_paths)} paths from previous iteration.")
    
    # Get candidate paths for this category
    candidate_paths = get_candidate_paths(feff_base_dir, feff_category)
    if not candidate_paths:
        print(f"  No candidate paths available for {group_id}, skipping.")
        return False, {}
    
    # Display some of the path options
    print(f"  Found {len(candidate_paths)} candidate paths for testing.")
    for i, p in enumerate(candidate_paths[:min(5, len(candidate_paths))]):
        print(f"    {i+1}: {p['label']} (reff = {p['reff']:.3f} Å)")
    if len(candidate_paths) > 5:
        print(f"    ... and {len(candidate_paths)-5} more paths")
    
    # Evaluate each candidate path
    test_results = evaluate_candidate_paths(
        group, temp_data, trans, prev_paths, candidate_paths, 
        prev_fit, feff_category
    )
    
    if not test_results:
        print(f"  No successful test fits for {group_id}, skipping.")
        return False, {}
    
    # Get the best candidate based on quality metric
    test_results.sort(key=lambda x: x['quality_metric'])
    best_candidate = test_results[0]
    print(f"\n  Best candidate for {group_id}: {best_candidate['label']}")
    print(f"    R-factor: {best_candidate['rfactor']:.6f}")
    print(f"    Quality metric: {best_candidate['quality_metric']:.6f}")
    
    # Perform final fit with the best candidate
    result_dict = perform_final_fit(
        group, group_id, temp_data, trans, prev_paths, 
        best_candidate, feff_category, iteration, prev_fit
    )
    
    return True, result_dict

In [7]:
def evaluate_candidate_paths(group, temp_data, trans, prev_paths, candidate_infos, 
                            prev_fit, feff_category):
    """
    Evaluate each candidate path by performing test fits.
    
    Parameters:
        group: The data group
        temp_data: Temporary data group for fitting
        trans: Transform parameters
        prev_paths: Paths from previous iteration
        candidate_infos: List of candidate path information
        prev_fit: Previous fit result
        feff_category: Category of paths being evaluated
        
    Returns:
        list: Test results for each candidate path
    """
    test_results = []
    print("  Evaluating candidate paths...")
    
    # Extract previous parameters
    if prev_fit is not None and hasattr(prev_fit, 'params'):
        prev_params = prev_fit.params
        amp_value = prev_params['amp'].value
        del_e0_value = prev_params['del_e0'].value
        n_PbI_value = prev_params['n_PbI'].value
        pbi_sig2_value = prev_params['pbi_sig2'].value
        pbi_delr_value = prev_params['pbi_delr'].value
    else:
        # Default values
        amp_value = 0.94
        del_e0_value = 0.0
        n_PbI_value = 3.0
        pbi_sig2_value = 0.01
        pbi_delr_value = 0.0
        
    cat_prefix = feff_category.lower().replace('-', '')
    
    for i, info in enumerate(candidate_infos):
        try:
            # Create a test path from this candidate
            fp_obj = info['feffpath']
            candidate_label = info.get('label', 'unknown')
            reff_candidate = info.get('reff', 0)
            
            # Setup test path
            test_path = feffpath(fp_obj.filename)
            test_path.degen = 1.0
            test_path.s02 = f'amp * n_{cat_prefix}'
            test_path.sigma2 = f'{cat_prefix}_sig2'
            test_path.deltar = f'{cat_prefix}_delr'
            test_path.e0 = 'del_e0'
            test_path.label = candidate_label
            
            # Combine with previous paths
            test_paths = copy.deepcopy(prev_paths)
            test_paths.append(test_path)
            
            # For test fit, fix previous parameters to isolate effect of new path
            test_params = param_group(
                amp = param(amp_value, vary=False),
                del_e0 = param(del_e0_value, vary=False),
                n_PbI = param(n_PbI_value, vary=False),
                pbi_sig2 = param(pbi_sig2_value, vary=False),
                pbi_delr = param(pbi_delr_value, vary=False),
                **{f"n_{cat_prefix}": param(1.0, vary=True, min=0.1, max=8.0),
                   f"{cat_prefix}_sig2": param(0.01, vary=True, min=0.001, max=0.03), 
                   f"{cat_prefix}_delr": param(0.0, vary=True, min=-0.2, max=0.2)}
            )
            
            # Create dataset and perform test fit
            dset = feffit_dataset(data=temp_data, pathlist=test_paths, transform=trans)
            test_fit = feffit(test_params, dset)
            
            # Calculate quality metrics
            delr_param = f"{cat_prefix}_delr"
            delr_abs = abs(test_fit.params[delr_param].value) if delr_param in test_fit.params else 0
            n_param = f"n_{cat_prefix}"
            n_value = test_fit.params[n_param].value if n_param in test_fit.params else 0
            
            # Quality metric combines R-factor and parameter reasonableness
            quality_metric = delr_abs * test_fit.rfactor
            
            test_results.append({
                'info': info,
                'path': test_path,
                'reff': reff_candidate,
                'rfactor': test_fit.rfactor,
                f"n_{cat_prefix}": n_value,
                f"{cat_prefix}_sig2": test_fit.params[f"{cat_prefix}_sig2"].value,
                f"{cat_prefix}_delr": test_fit.params[f"{cat_prefix}_delr"].value,
                'delr_abs': delr_abs,
                'quality_metric': quality_metric,
                'fit_result': test_fit,
                'label': candidate_label,
                'path_type': info.get('path_type', 'unknown')
            })
            
            print(f"    {i+1}: {candidate_label} - |ΔR|: {delr_abs:.3f}, " + 
                  f"R-factor: {test_fit.rfactor:.6f}, Quality: {quality_metric:.6f}")
                  
        except Exception as e:
            print(f"    Error testing candidate {i+1} ({info.get('label', 'unknown')}): {e}")
    
    return test_results

## Perform Final Fit and Parameter Refinement

After evaluating candidate paths, we'll perform the final fit with the best candidate and determine if it improves the model.

In [8]:
def perform_final_fit(group, group_id, temp_data, trans, prev_paths, 
                     best_candidate, feff_category, iteration, prev_fit):
    """
    Perform final fit with the best candidate path and determine if it improves the model.
    
    Parameters:
        group: Data group
        group_id: Group identifier
        temp_data: Temporary data group
        trans: Transform parameters
        prev_paths: Paths from previous iteration
        best_candidate: Best candidate path information
        feff_category: Path category
        iteration: Current iteration number
        prev_fit: Previous fit result
        
    Returns:
        dict: Results from the final fit
    """
    print(f"  Performing final fit with best candidate for {group_id}...")
    cat_prefix = feff_category.lower().replace('-', '')
    
    # Extract parameters from previous fit
    prev_rfactor = prev_fit.rfactor
    amp_value = prev_fit.params['amp'].value
    del_e0_value = prev_fit.params['del_e0'].value
    n_PbI_value = prev_fit.params['n_PbI'].value
    pbi_sig2_value = prev_fit.params['pbi_sig2'].value
    pbi_delr_value = prev_fit.params['pbi_delr'].value
    
    # Recreate candidate path
    best_info = best_candidate['info']
    best_fp_obj = best_info['feffpath']
    new_path = feffpath(best_fp_obj.filename)
    new_path.degen = 1.0
    new_path.s02 = f'amp * n_{cat_prefix}'
    new_path.sigma2 = f'{cat_prefix}_sig2'
    new_path.deltar = f'{cat_prefix}_delr'
    new_path.e0 = 'del_e0'
    new_path.label = best_candidate['label']
    
    # Combine with previous paths
    final_paths = copy.deepcopy(prev_paths)
    final_paths.append(new_path)
    
    # For final fit, allow all parameters to vary
    final_params = param_group(
        amp = param(amp_value, vary=False),
        del_e0 = param(del_e0_value, vary=True, min=-15, max=15),
        n_PbI = param(n_PbI_value, vary=True, min=0.0, max=8.0),
        pbi_sig2 = param(pbi_sig2_value, vary=True, min=0.001, max=0.03),
        pbi_delr = param(pbi_delr_value, vary=True, min=-0.2, max=0.2),
        **{f"n_{cat_prefix}": param(best_candidate[f"n_{cat_prefix}"], vary=True, min=0.1, max=8.0),
           f"{cat_prefix}_sig2": param(best_candidate[f"{cat_prefix}_sig2"], vary=True, min=0.001, max=0.03),
           f"{cat_prefix}_delr": param(best_candidate[f"{cat_prefix}_delr"], vary=True, min=-0.2, max=0.2)}
    )
    
    # Create dataset and perform final fit
    final_dset = feffit_dataset(data=temp_data, pathlist=final_paths, transform=trans)
    final_fit = feffit(final_params, final_dset)
    
    # Check if R-factor improved
    new_rfactor = final_fit.rfactor
    print(f"  Previous R-factor: {prev_rfactor:.6f}, New R-factor: {new_rfactor:.6f}")
    
    # Create result dictionary
    result = {
        'improved': new_rfactor < prev_rfactor,
        'fit_result': final_fit,
        'dataset': final_dset,
        'new_path': new_path,
        'paths': final_paths,
        'prev_rfactor': prev_rfactor,
        'new_rfactor': new_rfactor,
        'improvement': prev_rfactor - new_rfactor,
        'summary': {
            'group': group_id,
            'iteration': iteration,
            'category': feff_category,
            'best_path': new_path.label,
            'reff': new_path.reff,
            'prev_rfactor': prev_rfactor,
            'new_rfactor': new_rfactor,
            'improved': new_rfactor < prev_rfactor,
            'improvement': prev_rfactor - new_rfactor,
            'status': 'Accepted' if new_rfactor < prev_rfactor else 'Rejected'
        }
    }
    
    # If R-factor improved, store results on group
    if new_rfactor < prev_rfactor:
        print(f"  R-factor improved by {prev_rfactor - new_rfactor:.6f}, incorporating new path.")
        
        # Calculate effective distances for reporting
        pbi_r_eff = np.mean([p.reff for p in prev_paths if 'Pb-I' in p.label]) + final_fit.params['pbi_delr'].value
        new_r_eff = new_path.reff + final_fit.params[f"{cat_prefix}_delr"].value
        
        print(f"  Pb-I effective R: {pbi_r_eff:.3f} Å")
        print(f"  {feff_category} effective R: {new_r_eff:.3f} Å")
        
        # Store on group for future use
        setattr(group, f"fit_result_iter{iteration}", final_fit)
        setattr(group, f"paths_iter{iteration}", final_paths)
        group.__dict__[f'{cat_prefix}_path_iter{iteration}'] = new_path
        
        # Create k- and r-space data for visualization
        dset0 = final_dset
        k = dset0.data.k
        chi_data = dset0.data.chi * k**3
        chi_fit = dset0.model.chi * k**3
        r = dset0.data.r
        chir_data_mag = np.sqrt(dset0.data.chir_re**2 + dset0.data.chir_im**2)
        chir_fit_mag = np.sqrt(dset0.model.chir_re**2 + dset0.model.chir_im**2)
        
        group.__dict__[f'k_space_iter{iteration}'] = {'k': k, 'data': chi_data, 'fit': chi_fit}
        group.__dict__[f'r_space_iter{iteration}'] = {'r': r, 'data': chir_data_mag, 'fit': chir_fit_mag}
        
        # Update summary with additional information
        result['summary'].update({
            'n_PbI': final_fit.params['n_PbI'].value,
            'pbi_sig2': final_fit.params['pbi_sig2'].value,
            'pbi_delr': final_fit.params['pbi_delr'].value,
            'pbi_eff_r': pbi_r_eff,
            f"n_{cat_prefix}": final_fit.params[f"n_{cat_prefix}"].value,
            f"{cat_prefix}_sig2": final_fit.params[f"{cat_prefix}_sig2"].value,
            f"{cat_prefix}_delr": final_fit.params[f"{cat_prefix}_delr"].value,
            f"{cat_prefix}_eff_r": new_r_eff
        })
        
        # Create visualization
        visualize_fit_results(group, group_id, iteration, final_paths, new_path)
        
    else:
        print(f"  R-factor did not improve ({prev_rfactor:.6f} -> {new_rfactor:.6f}), keeping previous model.")
    
    return result

## Visualize Fit Results

Let's create functions to visualize our fit results in both k-space and R-space, with comparisons to previous iterations.

In [9]:
def visualize_fit_results(group, group_id, iteration, paths=None, new_path=None):
    """
    Visualize EXAFS fit results in k-space and R-space.
    
    Parameters:
        group: Data group with fit results
        group_id: Identifier for the group
        iteration: Current iteration number
        paths: List of paths used in the fit
        new_path: Newly added path to highlight
    """
    # Get fit result and datasets
    fit_result = getattr(group, f"fit_result_iter{iteration}", None)
    if fit_result is None:
        print(f"No fit result found for iteration {iteration}")
        return
    
    if not paths and hasattr(group, f"paths_iter{iteration}"):
        paths = getattr(group, f"paths_iter{iteration}")
    
    # Get k-space and R-space data from stored attributes
    k_space = getattr(group, f"k_space_iter{iteration}", None)
    r_space = getattr(group, f"r_space_iter{iteration}", None)
    
    # If data not stored, try to extract from fit_result
    if k_space is None or r_space is None:
        if hasattr(fit_result, 'datasets') and len(fit_result.datasets) > 0:
            dset = fit_result.datasets[0]
            k = dset.data.k
            chi_data = dset.data.chi * k**3
            chi_fit = dset.model.chi * k**3
            r = dset.data.r
            chir_data_mag = np.sqrt(dset.data.chir_re**2 + dset.data.chir_im**2)
            chir_fit_mag = np.sqrt(dset.model.chir_re**2 + dset.model.chir_im**2)
            k_space = {'k': k, 'data': chi_data, 'fit': chi_fit}
            r_space = {'r': r, 'data': chir_data_mag, 'fit': chir_fit_mag}
        else:
            print(f"Cannot extract fit data for visualization")
            return
    
    # Create visualization
    fig, (ax_k, ax_r) = plt.subplots(1, 2, figsize=(15, 6))
    
    # k-space plot
    ax_k.plot(k_space['k'], k_space['data'], 'b-', label='Data')
    ax_k.plot(k_space['k'], k_space['fit'], 'r--', label=f'Fit (Iter {iteration})')
    
    # Add previous iteration for comparison
    if iteration > 1:
        prev_k = getattr(group, f"k_space_iter{iteration-1}", None)
        if prev_k:
            ax_k.plot(prev_k['k'], prev_k['fit'], 'g-.', 
                     label=f'Fit (Iter {iteration-1})', alpha=0.5)
    
    # Add kmin/kmax markers if available
    if hasattr(fit_result, 'datasets') and len(fit_result.datasets) > 0:
        dset = fit_result.datasets[0]
        if hasattr(dset, 'transform'):
            kmin = getattr(dset.transform, 'kmin', None)
            kmax = getattr(dset.transform, 'kmax', None)
            if kmin:
                ax_k.axvline(kmin, color='gray', linestyle=':')
            if kmax:
                ax_k.axvline(kmax, color='gray', linestyle=':')
    
    ax_k.set_xlabel('$k$ (Å$^{-1}$)')
    ax_k.set_ylabel('$k^3\chi(k)$ (Å$^{-3}$)')
    ax_k.set_title(f'{group_id} - k-space (Iteration {iteration})')
    ax_k.legend()
    ax_k.grid(True, alpha=0.3)
    
    # R-space plot
    ax_r.plot(r_space['r'], r_space['data'], 'b-', label='Data')
    ax_r.plot(r_space['r'], r_space['fit'], 'r--', label=f'Fit (Iter {iteration})')
    
    # Add previous iteration
    if iteration > 1:
        prev_r = getattr(group, f"r_space_iter{iteration-1}", None)
        if prev_r:
            ax_r.plot(prev_r['r'], prev_r['fit'], 'g-.', 
                     label=f'Fit (Iter {iteration-1})', alpha=0.5)
    
    # Add rmin/rmax markers
    if hasattr(fit_result, 'datasets') and len(fit_result.datasets) > 0:
        dset = fit_result.datasets[0]
        if hasattr(dset, 'transform'):
            rmin = getattr(dset.transform, 'rmin', None)
            rmax = getattr(dset.transform, 'rmax', None)
            if rmin:
                ax_r.axvline(rmin, color='gray', linestyle=':')
            if rmax:
                ax_r.axvline(rmax, color='gray', linestyle=':')
    
    # Mark path positions on the R-space plot
    colors = ['green', 'magenta', 'cyan', 'orange', 'purple']
    path_labels = {}
    
    for i, path in enumerate(paths or []):
        if not hasattr(path, 'reff'):
            continue
            
        reff = path.reff
        path_type = 'unknown'
        color_idx = 0
        
        # Determine path type and color
        if 'Pb-I' in path.label:
            path_type = 'Pb-I'
            color_idx = 0
        elif 'Pb-O' in path.label:
            path_type = 'Pb-O'
            color_idx = 1
        elif 'Pb-S' in path.label:
            path_type = 'Pb-S'
            color_idx = 2
        else:
            path_type = path.label[:4]
            color_idx = i % len(colors)
            
        color = colors[color_idx]
        
        # Highlight newly added path
        alpha = 0.6
        linewidth = 1
        if new_path and path.label == new_path.label:
            alpha = 1.0
            linewidth = 2
            
        # Only add line and label if we haven't seen this path type yet
        if path_type not in path_labels:
            ax_r.axvline(reff, color=color, linestyle='--', alpha=alpha, linewidth=linewidth)
            
            # Position the label at 80% of the y-axis height
            y_pos = 0.8 * ax_r.get_ylim()[1] * (0.9 - 0.1*color_idx)
            
            ax_r.text(reff, y_pos, path.label, ha='center', rotation=90, 
                    alpha=alpha, fontsize=8, color=color,
                    bbox=dict(facecolor='white', alpha=0.5, pad=1))
            
            path_labels[path_type] = True
    
    ax_r.set_xlabel('$R$ (Å)')
    ax_r.set_ylabel('$|\chi(R)|$ (Å$^{-3}$)')
    ax_r.set_title(f'{group_id} - R-space (Iteration {iteration})')
    ax_r.legend()
    ax_r.grid(True, alpha=0.3)
    
    # Add fit info text box
    if hasattr(fit_result, 'rfactor'):
        # Get improvement from previous iteration
        improvement = ""
        if iteration > 1:
            prev_fit = getattr(group, f"fit_result_iter{iteration-1}", None)
            if prev_fit and hasattr(prev_fit, 'rfactor'):
                prev_rfactor = prev_fit.rfactor
                improvement = f" (Δ = {prev_rfactor - fit_result.rfactor:.6f})"
        
        # Prepare info text
        params = fit_result.params
        info_text = [
            f"R-factor: {fit_result.rfactor:.6f}{improvement}",
            f"χ²: {fit_result.chi_square:.2f}",
        ]
        
        # Add k-range
        if hasattr(fit_result, 'datasets') and fit_result.datasets:
            dset = fit_result.datasets[0]
            if hasattr(dset, 'transform'):
                kmin = getattr(dset.transform, 'kmin', None)
                kmax = getattr(dset.transform, 'kmax', None)
                if kmin is not None and kmax is not None:
                    info_text.append(f"k-range: {kmin:.1f}-{kmax:.1f} Å⁻¹")
        
        # Add parameter values
        info_text.append("\nParameters:")
        for p_name in params:
            p = params[p_name]
            stderr = f" ± {p.stderr:.4f}" if p.stderr else ""
            info_text.append(f"{p_name}: {p.value:.4f}{stderr}")
        
        # Add the text box
        ax_r.text(0.05, 0.95, '\n'.join(info_text), 
                transform=ax_r.transAxes, fontsize=8,
                verticalalignment='top', 
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    
    plt.tight_layout()
    plt.show()
    
    return fig

## Analyze and Compare Results Across Iterations

Finally, let's create functions to analyze results across iterations, allowing us to track improvements and changes in our EXAFS model.

In [10]:
def analyze_iterations(xasmu2r, group_id, max_iteration=5):
    """
    Analyze fitting results across iterations for a specific group.
    
    Parameters:
        xasmu2r: XASmu2r instance with fitted results
        group_id: Group identifier (e.g. "proj_name.group_name")
        max_iteration: Maximum iteration to check
        
    Returns:
        pandas.DataFrame: Summary of parameters across iterations
    """
    # Parse the group identifier and get the group
    parts = group_id.split(".")
    if len(parts) != 2:
        print(f"Invalid group ID format: {group_id}. Expected format: project.group")
        return None
    
    proj_name, group_name = parts
    
    # Check if project exists
    if proj_name not in xasmu2r.projects:
        print(f"Project '{proj_name}' not found!")
        return None
    
    # Check if group exists
    if group_name not in xasmu2r.projects[proj_name].groups:
        print(f"Group '{group_name}' not found in project '{proj_name}'!")
        return None
    
    # Get the group
    group = xasmu2r.projects[proj_name].groups[group_name]
    
    # Collect data for each iteration
    iterations = []
    
    for i in range(1, max_iteration + 1):
        fit_attr = f"fit_result_iter{i}"
        paths_attr = f"paths_iter{i}"
        
        if not hasattr(group, fit_attr):
            continue
            
        fit_result = getattr(group, fit_attr)
        paths = getattr(group, paths_attr, [])
        
        # Basic iteration info
        iter_data = {
            'iteration': i,
            'rfactor': fit_result.rfactor,
            'chi_square': fit_result.chi_square,
            'n_paths': len(paths),
        }
        
        # Add parameter values
        for p_name, p in fit_result.params.items():
            iter_data[f"{p_name}"] = p.value
            if p.stderr:
                iter_data[f"{p_name}_err"] = p.stderr
        
        # Add path information
        path_types = {}
        for path in paths:
            if 'Pb-I' in path.label:
                path_types['Pb-I'] = path_types.get('Pb-I', 0) + 1
            elif 'Pb-O' in path.label:
                path_types['Pb-O'] = path_types.get('Pb-O', 0) + 1
            elif 'Pb-S' in path.label:
                path_types['Pb-S'] = path_types.get('Pb-S', 0) + 1
            else:
                path_types['other'] = path_types.get('other', 0) + 1
                
        for path_type, count in path_types.items():
            iter_data[f"n_{path_type.replace('-', '')}"] = count
        
        iterations.append(iter_data)
    
    # Create DataFrame
    if not iterations:
        print(f"No fit results found for {group_id}")
        return None
        
    return pd.DataFrame(iterations)

def visualize_iteration_progress(df):
    """
    Visualize how parameters change across iterations.
    
    Parameters:
        df: DataFrame from analyze_iterations()
    """
    if df is None or len(df) == 0:
        print("No data to visualize")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Plot R-factor
    axes[0, 0].plot(df['iteration'], df['rfactor'], 'o-', color='blue')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('R-factor')
    axes[0, 0].set_title('R-factor vs. Iteration')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot coordination numbers if available
    coord_params = [col for col in df.columns if col.startswith('n_') and not col.endswith('err')]
    
    if coord_params:
        for param in coord_params:
            if param in df.columns:
                axes[0, 1].plot(df['iteration'], df[param], 'o-', label=param)
        
        axes[0, 1].set_xlabel('Iteration')
        axes[0, 1].set_ylabel('Coordination Number')
        axes[0, 1].set_title('Coordination Numbers vs. Iteration')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
    
    # Plot Debye-Waller factors if available
    sig2_params = [col for col in df.columns if 'sig2' in col and not col.endswith('err')]
    
    if sig2_params:
        for param in sig2_params:
            if param in df.columns:
                axes[1, 0].plot(df['iteration'], df[param], 'o-', label=param)
        
        axes[1, 0].set_xlabel('Iteration')
        axes[1, 0].set_ylabel('σ² (Å²)')
        axes[1, 0].set_title('Debye-Waller Factors vs. Iteration')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    
    # Plot distance shifts if available
    delr_params = [col for col in df.columns if 'delr' in col and not col.endswith('err')]
    
    if delr_params:
        for param in delr_params:
            if param in df.columns:
                axes[1, 1].plot(df['iteration'], df[param], 'o-', label=param)
        
        axes[1, 1].set_xlabel('Iteration')
        axes[1, 1].set_ylabel('ΔR (Å)')
        axes[1, 1].set_title('Distance Shifts vs. Iteration')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Using the Iterative Fitting Workflow

Now let's demonstrate how to use the functions we've created to perform iterative EXAFS fitting. The workflow typically follows these steps:

1. Load XAS data from Athena project files
2. Prepare the data (background subtraction, normalization, etc.)
3. Perform Iteration 1 to find the best Pb-I path
4. Perform Iteration 2 to add a Pb-O path
5. Perform Iteration 3 to add a Pb-S path (if appropriate)
6. Analyze the results across iterations

Here's a complete example:

In [13]:
# Example usage of the iterative EXAFS fitting workflow

# 1. Initialize the class and load projects
prj_folder =  r"C:\Users\kwill\Keenan_UCB-O365\OneDrive - UCB-O365\Data\XAS\AsminData\Athenprjcalibrated_L3\Athenprjcalibrated_L3\Pb_edge"
xas = XASmu2r_Lite(prj_folder)
xas.load_projects()

# 2. Set up FEFF path directories
feff_base_dir = r"C:/Users/kwill/.larch/feff/relaxed"

# 3. Run Iteration 1 - Finding best Pb-I path
#This step would typically be done with a separate function like:
iteration1_results = enhanced_iteration1_fit(xas, feff_base_dir)


Loading project file: S10_0p1_PbBr2_MAF_calibrated_merged.prj
Loading project file: S11_0p1_PBBR2_MAAc_calibrated_PbL3.prj
Loading project file: S14_0p3_PbBr2_MAAC_calibrated_PbL3.prj
Loading project file: S15_0p3_PbBr2_MAP_calibrated_PbL3.prj
Loading project file: S17_0p5_PbBr2_MAAc_calibrated_PbL3.prj
Loading project file: S18_0p5_PbBr2_MAP_calibrated_PbL3.prj
Loading project file: S1_0P1_PbI2_MAF_PbEdge.prj
Loading project file: S4_0p3_PbI2_MAF_calibrated.prj
Loading project file: S6_PbI2_0p3_MAP_calibrated.prj

Available Groups in Project 'S10_0p1_PbBr2_MAF_calibrated_merged':
  S10_0p1_PbBr2_MAF_pos24_merge_8_PbEdge

Available Groups in Project 'S11_0p1_PBBR2_MAAc_calibrated_PbL3':
  S11_0p1_PbBr2_MAAc_pos24_merge_9_clean_PbL3

Available Groups in Project 'S14_0p3_PbBr2_MAAC_calibrated_PbL3':
  S14_0p3_PbBr2_MAAc_pos24_merge_9_clean_PbL3

Available Groups in Project 'S15_0p3_PbBr2_MAP_calibrated_PbL3':
  S15_0p3_PbBr2_MAP_pos24_merge_9_clean_PbL3

Available Groups in Project 'S17_

TypeError: object of type 'XASmu2r_Lite' has no len()

In [ ]:

# 4. Run Iteration 2 - Adding Pb-O paths
iteration = 2
feff_category = "Pb-O"
params_iter2 = iterative_fit_all(xas, feff_category, iteration, feff_base_dir)

# 5. Run Iteration 3 - Adding Pb-S paths
iteration = 3
feff_category = "Pb-S"
params_iter3 = iterative_fit_all(xas, feff_category, iteration, feff_base_dir, 
                               previous_params=params_iter2)

# 6. Analyze results for a specific group
group_id = "example_project.sample1"  # Replace with actual group id
iter_df = analyze_iterations(xas, group_id)
if iter_df is not None:
    display(iter_df)
    visualize_iteration_progress(iter_df)